# HW06 Data Science Academy
# Мурзабеков Эльдар 

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.

In [277]:
import pandas as pd 

1) Создаем series объект со значением 1, None, 3. С помощью isna считаем количество пустых значений и суммируем с помощью sum. В первом выводе будет 1, затем заполняем пустые значения нулем с помощью fillna и считаем среднее, будет 1.33

In [278]:
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())

1 1.3333333333333333


2) К series применяем strip для удаления пробелов по краям, lower для преобразования в строчный вид и с помощью nunique, считаем количество уникальных элементов, будет 2

In [279]:
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())

2


3) С помощью drop_duplicates удаляем дубликаты в датафрейме создав новый, не меняя исходный, но вывод будет 3, так-как удаление дубликатов не присвоили с помощью =

In [280]:
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))

3


4) replace удаляет пробелы, затем to_numeric превращает содержимый текст в числа. Вывод суммируем с sum, будет 1005

In [281]:
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())

1005


5) datetime преобразует строки во временной формат, с помощью dt.month и tolist возвращает месяцы в списке [9, 10]

In [282]:
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

[9, 10]


In [283]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [284]:
def clean(path):
    df = pd.read_csv(path)
    print(f"Подсчет после загрузки: {len(df)}")
    df['amount'] = pd.to_numeric(df['amount'].str.replace(' ', ''))
    print(f"После преобразования amount: {len(df)}")
    city_normal = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
    df['city'] = df['city'].str.strip().str.lower().map(city_normal)
    print(f"После преобразования city: {len(df)}")
    df = df.drop_duplicates()
    print(f"После удаления дубликатов: {len(df)}")
    df['order_date'] = pd.to_datetime(df['order_date'])
    print(f"После преобразования order_date: {len(df)}")
    df = df[df['customer_id'] != 'TEST']
    print(f"После удаления пользователя TEST: {len(df)}")
    print(f"Проверка NaN amount: {df['amount'].isna().sum()}")
    print(f"Проверка NaN city: {df['city'].isna().sum()}")
    print(f"Проверка order_id на уникальность: {df['order_id'].is_unique}")
    return df 
clean_df = clean('orders.csv')
print(f"Результат строк: {len(clean_df)}")

Подсчет после загрузки: 5463
После преобразования amount: 5463
После преобразования city: 5463
После удаления дубликатов: 5283
После преобразования order_date: 5283
После удаления пользователя TEST: 5280
Проверка NaN amount: 0
Проверка NaN city: 0
Проверка order_id на уникальность: True
Результат строк: 5280


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [285]:
sept = clean_df['order_date'].dt.month == 9
octo = clean_df['order_date'].dt.month == 10
paid = clean_df['status'] == 'оплачен'
revenue_sep = clean_df['amount'][sept & paid].sum()
revenue_oct = clean_df['amount'][octo & paid].sum() 
revenue_change = (revenue_oct - revenue_sep) / revenue_sep * 100
print(f"Выручка оплаченных заказов за сентябрь: {revenue_sep}")
print(f"Выручка оплаченных заказов за октябрь: {revenue_oct}")
print(f"Изменение: {revenue_change:.2f}%")
for city in clean_df['city'].unique():
    city_sep = clean_df['amount'][((clean_df['city'] == city) & sept & paid)].sum()
    city_oct = clean_df['amount'][((clean_df['city'] == city) & octo & paid)].sum()
    change = (city_oct - city_sep) / city_sep * 100
    print(f"Выручка {city} за сентябрь: {city_sep}")
    print(f"Выручка {city} за октябрь: {city_oct}")
    print(f"Изменение {city}: {change:.2f}%")


Выручка оплаченных заказов за сентябрь: 35243410.0
Выручка оплаченных заказов за октябрь: 34419520.0
Изменение: -2.34%
Выручка Алматы за сентябрь: 14501070.0
Выручка Алматы за октябрь: 14643020.0
Изменение Алматы: 0.98%
Выручка Астана за сентябрь: 9034220.0
Выручка Астана за октябрь: 8850530.0
Изменение Астана: -2.03%
Выручка Караганда за сентябрь: 3648520.0
Выручка Караганда за октябрь: 3371680.0
Изменение Караганда: -7.59%
Выручка Шымкент за сентябрь: 4558650.0
Выручка Шымкент за октябрь: 4044010.0
Изменение Шымкент: -11.29%
Выручка Атырау за сентябрь: 923990.0
Выручка Атырау за октябрь: 969860.0
Изменение Атырау: 4.96%
Выручка Актобе за сентябрь: 2576960.0
Выручка Актобе за октябрь: 2540420.0
Изменение Актобе: -1.42%


В октябре общая выручка сократилась на 2,34% по сравнению с сентябрём. Как и показал анализ в ДЗ 4, наибольшее влияние на снижение оказал Шымкент, на который пришлось около 62,5% общего падения. Выручка в городе уменьшилась на 11,29%. Заметное снижение также наблюдается в Караганде, где выручка снизилась на 7,59%.

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

| Этап очистки | Удалено строк | Осталось строк | Изменение выручки октябрь/сентябрь |
|---|---:|---:|---:|
| Исходные данные | 0 | 5 463 | −22,7% |
| Удаление TEST | 3 | 5 460 | −8,2% |
| Удаление дубликатов | 180 | 5 280 | −2,34% |

После очистки данных количество записей сократилось на 183, из которых 180 оказались дубликатами, а ещё 3 были тестовыми записями. Наибольшее влияние на искажение результатов оказали именно тестовые записи, после их исключения падение выручки сократилось с 22,7% до 8,2%. Удаление дубликатов дополнительно уменьшило падение выручки до 2,34%. Дубликаты также завысили выручку за сентябрь, поэтому при анализе выручки важно исключать как тестовые записи, так и дубликаты. P.S. Шаблон Markdown-таблицы взял из исходника урока :)

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [286]:
print(f"Пропуски delivery_days по чистым данным: {clean_df['delivery_days'].isna().sum()}")
print(f"Статусы: {clean_df.loc[clean_df['delivery_days'].isna(), 'status'].value_counts()}")
print(f"Распределение по городам: {clean_df[clean_df['delivery_days'].isna()]['city'].value_counts()}")
sh = clean_df.loc[clean_df['city'] == 'Шымкент', 'delivery_days']
print(f"dropna: {(sh.dropna() <= 2).mean():.1%}")
print(f"fillna медиана: {(sh.fillna(sh.median()) <= 2).mean():.1%}")
print(f"fillna 0: {(sh.fillna(0) <= 2).mean():.1%}")

Пропуски delivery_days по чистым данным: 158
Статусы: status
оплачен    144
отменён     14
Name: count, dtype: int64
Распределение по городам: city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64
dropna: 29.8%
fillna медиана: 29.0%
fillna 0: 31.8%


Честным вариантом является dropna(), так как он не создаёт искусственные данные, как fillna() при заполнении пропусков медианой или нулём. dropna() может удалить некоторые строки с пропущенными значениями, но зато не придумывает данные, которых изначально не было. При этом результат будет учитывать только заказы, у которых известен срок доставки.

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [287]:
df = pd.read_csv('orders.csv')
df = df.drop_duplicates()
c = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
df['city'] = df['city'].str.strip().str.lower().map(c)
print(df['city'].nunique(), len(df))

6 5283


Ошибки которые были исправлены:

1) drop_duplicates не был присвоен обратно df, поэтому дубликаты оставались в основной таблице

2) Использовался только lower без strip, из-за чего пробелы по краям названий городов сохранялись. Также добавил нормализацию названий городов, чтобы русские и английские варианты считались одинаковыми.

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [288]:
print(f"Заказы по дням: {clean_df['order_date'].dt.day_name().value_counts()}")
days = clean_df['order_date'].dt.day_name().value_counts()
print(f"Больше всего заказов: {max(days)} {days.index[0]}")
print(f"Меньше всего заказов: {min(days)} {days.index[-1]}")
print(f"Разница заказов между ними: {max(days) - min(days)} ({((max(days) - min(days)) / max(days)) * 100:.2f}%)")

Заказы по дням: order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64
Больше всего заказов: 786 Thursday
Меньше всего заказов: 684 Monday
Разница заказов между ними: 102 (12.98%)


Больше всего заказов приходится на четверг (786), меньше всего на понедельник (684). Разница между этими днями составляет 102 заказа (12,98%). При составлении графика работы курьеров важно учитывать, что со вторника по субботу заказов больше, особенно в четверг. В понедельник нагрузка ниже, поэтому в этот день можно сократить количество смен или задействовать меньше курьеров.

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [289]:
sh_sep = ((clean_df['city'] == 'Шымкент') & (clean_df['order_date'].dt.month == 9))
sh_oct = ((clean_df['city'] == 'Шымкент') & (clean_df['order_date'].dt.month == 10))
print(f"Средний срок доставки в сентябре по Шымкенту: {clean_df['delivery_days'][sh_sep].mean():.1f}")
print(f"Средний срок доставки в октябре по Шымкенту: {clean_df['delivery_days'][sh_oct].mean():.1f}")
sh5 = clean_df['order_date'][(clean_df['city'] == 'Шымкент') & (clean_df['delivery_days'] >= 5)].sort_values()
sh_after = clean_df[(clean_df['city'] == 'Шымкент') & (clean_df['order_date'].dt.date >= sh5.iloc[0].date())]
print(f"Первая дата, где срок доставки не бывает меньше 5 дней: {clean_df['order_date'][sh5.index[0]]}")
print(f"Заказы в Шымкенте со сроком доставки меньше 5 дней после этой даты: {(sh_after['delivery_days'] < 5).sum()}")
others = clean_df[(clean_df['city'] != 'Шымкент') & (clean_df['order_date'].dt.date == sh5.iloc[0].date())]
print(f"Максимальный срок доставки в других городах в эту дату: {others['delivery_days'].max()}")

Средний срок доставки в сентябре по Шымкенту: 2.0
Средний срок доставки в октябре по Шымкенту: 6.5
Первая дата, где срок доставки не бывает меньше 5 дней: 2026-10-01 10:00:00
Заказы в Шымкенте со сроком доставки меньше 5 дней после этой даты: 0
Максимальный срок доставки в других городах в эту дату: 3.0


В Шымкенте средний срок доставки увеличился с 2 до 6,5 дня. Первые заказы с доставкой от 5 дней появились 1 октября. При этом в остальных городах максимальный срок доставки на эту дату составлял 3 дня. Это говорит о том, что проблема затрагивает именно Шымкент. Стоит проверить работу склада, наличие курьеров и изменения в логистических маршрутах с 1 октября

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [290]:
def to_amount(s):
    if type(s) == pd.Series:
        return pd.to_numeric(s.str.replace(' ', '').str.replace(',', '.'), errors='coerce')

eight = pd.read_csv('orders.csv')
print(to_amount(pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])))
print(f"Сумма: {to_amount(eight['amount']).sum()}")
print(f"Сумма урока исходного orders: {to_amount(pd.read_csv('orders.csv')['amount']).sum()}") 

0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64
Сумма: 109883360.0
Сумма урока исходного orders: 109883360.0


Для проверки функции использовал исходный orders.csv из урока, поскольку clean_df уже содержит очищенные данные. Поэтому суммы исходного и очищенного датафрейма могут отличаться. После преобразования значений в исходном файле получил сумму 109883360